In [1]:


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import snntorch as snn
from snntorch import spikegen

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên thiết bị: {device}")

Đang chạy trên thiết bị: cuda


In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset  
from sklearn.feature_extraction.text import TfidfVectorizer

vocab_size = 5000       
batch_size = 32         
num_steps = 20         
num_epochs = 15         
learning_rate = 1e-3

df_train = pd.read_csv("train_data_final.csv") 
df_test = pd.read_csv("test_data_final.csv")

x_train = df_train['text'].values
y_train = df_train['label'].values
x_test = df_test['text'].values
y_test = df_test['label'].values

vectorizer = TfidfVectorizer(max_features=vocab_size, stop_words='english')

X_train_sparse = vectorizer.fit_transform(x_train)

class SparseSpamDataset(Dataset):
    def __init__(self, sparse_matrix, labels):
        self.sparse_matrix = sparse_matrix
        self.labels = labels

    def __len__(self):
        return self.sparse_matrix.shape[0]

    def __getitem__(self, idx):
        dense_row = self.sparse_matrix[idx].toarray()[0]
        
        x_tensor = torch.tensor(dense_row, dtype=torch.float32)
        y_tensor = torch.tensor(self.labels[idx], dtype=torch.long)
        return x_tensor, y_tensor

dataset = SparseSpamDataset(X_train_sparse, y_train)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"Số lượng email dùng để train: {len(dataset)}")

Số lượng email dùng để train: 60641


In [3]:
import torch
import torch.nn as nn
import snntorch as snn

class SpamFilterSNN(nn.Module):

    def __init__(self, vocab_size=5000, hidden_size=128, num_classes=2, beta=0.9):
        super().__init__()
        self.fc1 = nn.Linear(vocab_size, hidden_size)
        self.lif1 = snn.Leaky(beta=beta)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        self.lif2 = snn.Leaky(beta=beta)

    def forward(self, x):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        
        spk2_rec = []
        mem2_rec = []

        num_steps = x.size(0) 

        for step in range(num_steps):
            # Tính toán lớp 1
            cur1 = self.fc1(x[step])
            spk1, mem1 = self.lif1(cur1, mem1)
            
            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            
            spk2_rec.append(spk2)
            mem2_rec.append(mem2)

        return torch.stack(spk2_rec, dim=0), torch.stack(mem2_rec, dim=0)

In [4]:
model = SpamFilterSNN(vocab_size=vocab_size).to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.999))

In [5]:
X_test_sparse = vectorizer.transform(x_test)
test_dataset = SparseSpamDataset(X_test_sparse, y_test)

test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

model = SpamFilterSNN(vocab_size=vocab_size).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.999))

loss_history = []
test_acc_history = []

print("🚀 Bắt đầu huấn luyện và kiểm tra trực tiếp trên tập Test...")

for epoch in range(num_epochs):
    
    model.train() 
    total_loss = 0

    for batch_idx, (data, targets) in enumerate(dataloader): 
        data, targets = data.to(device), targets.to(device)

        spike_data = spikegen.rate(data, num_steps=num_steps)
        optimizer.zero_grad()
        spk_out, mem_out = model(spike_data)
        total_spikes = spk_out.sum(dim=0) 
        
        loss = loss_fn(total_spikes, targets)
        loss.backward()  
        optimizer.step()  

        total_loss += loss.item()

    avg_train_loss = total_loss / len(dataloader)
    
    model.eval() 
    test_correct = 0
    test_total = 0
    
    with torch.no_grad(): 
        for data, targets in test_dataloader:
            data, targets = data.to(device), targets.to(device)
            
            spike_data = spikegen.rate(data, num_steps=num_steps)
            spk_out, _ = model(spike_data)
            total_spikes = spk_out.sum(dim=0)
            
            _, predicted = total_spikes.max(1)
            test_total += targets.size(0)
            test_correct += predicted.eq(targets).sum().item()

    test_acc = 100. * test_correct / test_total
    
    loss_history.append(avg_train_loss)
    test_acc_history.append(test_acc)

    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] - Train Loss: {avg_train_loss:.4f} - TEST ACCURACY: {test_acc:.2f}%")

print("✅ Hoàn tất huấn luyện!")

🚀 Bắt đầu huấn luyện và kiểm tra trực tiếp trên tập Test...
Epoch [01/15] - Train Loss: 0.1223 - TEST ACCURACY: 96.91%
Epoch [02/15] - Train Loss: 0.0655 - TEST ACCURACY: 97.03%
Epoch [03/15] - Train Loss: 0.0477 - TEST ACCURACY: 97.74%
Epoch [04/15] - Train Loss: 0.0327 - TEST ACCURACY: 97.87%
Epoch [05/15] - Train Loss: 0.0225 - TEST ACCURACY: 98.02%
Epoch [06/15] - Train Loss: 0.0160 - TEST ACCURACY: 98.23%
Epoch [07/15] - Train Loss: 0.0122 - TEST ACCURACY: 98.28%
Epoch [08/15] - Train Loss: 0.0107 - TEST ACCURACY: 98.21%
Epoch [09/15] - Train Loss: 0.0094 - TEST ACCURACY: 98.15%
Epoch [10/15] - Train Loss: 0.0091 - TEST ACCURACY: 98.39%
Epoch [11/15] - Train Loss: 0.0070 - TEST ACCURACY: 98.53%
Epoch [12/15] - Train Loss: 0.0062 - TEST ACCURACY: 98.27%
Epoch [13/15] - Train Loss: 0.0056 - TEST ACCURACY: 98.59%
Epoch [14/15] - Train Loss: 0.0065 - TEST ACCURACY: 98.48%
Epoch [15/15] - Train Loss: 0.0049 - TEST ACCURACY: 98.44%
✅ Hoàn tất huấn luyện!


In [ ]:
torch.save(model.state_dict(), "snn_model.pth")
with open("tfidf_vectorizer.pkl", "wb") as f:
    import pickle
    pickle.dump(vectorizer, f)